<a id="0"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#255E58;overflow:hidden">Análise Avançada de Dados para IoT  📊📈</div>

<a id="1"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden">Módulo 14 → Insights e Indicadores Estratégicos
</div>

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Importando
Bibliotecas</b></span>

In [ ]:
# BIBLIOTECAS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Wedge


# CONFIGURAÇÕES DE ESTÉTICA👇🏽

# Ajustes visuais do Jupyter
from IPython.core.display import HTML
display(HTML('<style>.container { width:80% !important; }</style>'))
# pd.set_option('display.max_columns', 60)

# Configurando pandas para mostrar todas as linhas e colunas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 

# Configurando pandas para não mostrar notação científica para números
pd.set_option('display.float_format', lambda x: '%.2f' % x)
# Serve para Manter os gráficos nesse mesmo Notebook
%matplotlib inline
# Ignorando alertas
import warnings
warnings.filterwarnings("ignore")

# Reprodutibilidade
np.random.seed(42)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 1

In [ ]:
def gerar_dados():
    np.random.seed(42)

    horas = pd.date_range("2026-10-03", periods=24, freq="1h")
    config = {
        "Câmara A": (4.2, 18),
        "Câmara B": (4.6, 20),
        "Câmara C": (6.4, 28)
    }

    dados = []
    for camara, (temp, energia) in config.items():
        for hora in horas:
            dados.append([
                hora,
                camara,
                temp + np.random.normal(0, 0.35),
                energia + np.random.normal(0, 1.2)
            ])

    return pd.DataFrame(
        dados,
        columns=["hora", "camara", "temperatura", "energia_kwh"]
    )


def grafico(valores, titulo, ylabel, cores, sufixo="", limite=None):
    plt.figure(figsize=(7, 4))
    barras = plt.bar(valores.index, valores, color=cores)

    if limite:
        plt.axhline(limite, color="black", linestyle="--", label="Limite = 5 °C")
        plt.legend()

    plt.bar_label(barras, labels=[f"{v:.2f}{sufixo}" for v in valores], padding=3)
    plt.title(titulo)
    plt.ylabel(ylabel)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


df = gerar_dados()

resumo = df.groupby("camara").agg(
    temperatura_media=("temperatura", "mean"),
    energia_media=("energia_kwh", "mean")
).round(2)

resumo["acima_limite"] = resumo["temperatura_media"] > 5

grafico(
    resumo["temperatura_media"],
    "Temperatura Média por Câmara",
    "Temperatura média (°C)",
    ["steelblue", "seagreen", "tomato"],
    " °C",
    5
)

grafico(
    resumo["energia_media"],
    "Consumo Médio de Energia por Câmara",
    "Energia média (kWh)",
    ["goldenrod", "mediumpurple", "teal"]
)

print(resumo)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 2

In [ ]:
def gerar_dados(n=1000):
    np.random.seed(42)
    return (
        np.random.normal(120, 25, n),
        np.random.rand(n) > 0.03,
        np.random.rand(n) > 0.02,
        np.random.rand(n) > 0.015
    )


latencia, recebida, valida, online = gerar_dados()

df_kpi = pd.DataFrame({
    "KPI": ["Disponibilidade (%)", "Entrega (%)", "Dados válidos (%)", "Latência média (ms)"],
    "Valor": [online.mean()*100, recebida.mean()*100, valida.mean()*100, latencia.mean()]
})

# 1. KPIs
pct = df_kpi.iloc[:3]

plt.figure(figsize=(8, 4))
barras = plt.bar(pct["KPI"], pct["Valor"], color=["green", "blue", "orange"])
plt.bar_label(barras, labels=[f"{v:.1f}%" for v in pct["Valor"]], padding=3)
plt.ylim(90, 100)
plt.title("KPIs de Confiabilidade do Sistema IoT")
plt.ylabel("Percentual (%)")
plt.grid(axis="y", alpha=0.3)
plt.show()

# 2. LATÊNCIA
plt.figure(figsize=(8, 4))
plt.hist(latencia, bins=20, color="purple", edgecolor="white")
plt.axvline(latencia.mean(), color="red", linestyle="--", linewidth=2,
            label=f"Média = {latencia.mean():.1f} ms")
plt.title("Distribuição da Latência das Mensagens")
plt.xlabel("Latência (ms)")
plt.ylabel("Quantidade de mensagens")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

# 3. RESULTADOS
print(df_kpi.round(2))

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 3

In [ ]:
df = pd.DataFrame({
    "maquina": ["M1", "M2", "M3"],
    "disponibilidade_pct": [99.2, 96.5, 94.8],
    "latencia_ms": [85, 130, 170],
    "producao_unid_h": [120, 105, 90],
    "custo_parada_rs": [800, 1800, 3200]
})


def grafico(coluna, titulo, ylabel, cores, formato, ylim=None):
    plt.figure(figsize=(8, 4))
    barras = plt.bar(df["maquina"], df[coluna], color=cores)

    plt.bar_label(
        barras,
        labels=[formato(v) for v in df[coluna]],
        padding=3
    )

    plt.title(titulo)
    plt.ylabel(ylabel)
    plt.grid(axis="y", alpha=0.3)

    if ylim:
        plt.ylim(ylim)

    plt.tight_layout()
    plt.show()


grafico(
    "disponibilidade_pct",
    "Indicador Técnico: Disponibilidade",
    "Disponibilidade (%)",
    ["green", "orange", "red"],
    lambda v: f"{v:.1f}%",
    (90, 100)
)

grafico(
    "custo_parada_rs",
    "Indicador de Negócio: Custo de Parada",
    "Custo de parada (R$)",
    ["purple", "blue", "brown"],
    lambda v: f"R$ {v}"
)

print(df)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 4

In [ ]:
df = pd.DataFrame({
    "maquina": ["M1", "M2", "M3", "M4", "M5"],
    "temperatura": [76, 84, 89, 81, 87],
    "criticidade_ativo": [1, 3, 2, 3, 1]
})


def severidade(t):
    return ("Baixa", 1) if t < 80 else ("Média", 2) if t < 86 else ("Alta", 3)


def prioridade(s):
    return "P1 - Imediata" if s >= 8 else "P2 - Alta" if s >= 6 else "P3 - Média" if s >= 4 else "P4 - Baixa"


df[["severidade", "peso_severidade"]] = df["temperatura"].apply(
    lambda x: pd.Series(severidade(x))
)

df["score_prioridade"] = df["peso_severidade"] * 2 + df["criticidade_ativo"]
df["prioridade"] = df["score_prioridade"].apply(prioridade)

# 1. SEVERIDADE
cores = {"Baixa": "green", "Média": "orange", "Alta": "red"}

plt.figure(figsize=(8, 4))
barras = plt.bar(df["maquina"], df["temperatura"],
                 color=[cores[s] for s in df["severidade"]])

plt.axhline(80, color="orange", linestyle="--", label="Média ≥ 80 °C")
plt.axhline(86, color="red", linestyle="--", label="Alta ≥ 86 °C")
plt.bar_label(barras, labels=df["severidade"], padding=3)
plt.title("Severidade dos Alertas por Temperatura")
plt.xlabel("Máquina")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()

# 2. PRIORIDADE
ordem = df.sort_values("score_prioridade", ascending=False)

plt.figure(figsize=(8, 4))
barras = plt.bar(
    ordem["maquina"],
    ordem["score_prioridade"],
    color=["red", "purple", "orange", "blue", "green"]
)

plt.bar_label(barras, labels=ordem["prioridade"], padding=3, fontsize=9)
plt.title("Prioridade de Atendimento dos Alertas")
plt.xlabel("Máquina")
plt.ylabel("Score de prioridade")
plt.grid(axis="y", alpha=0.3)
plt.show()

# 3. RESULTADOS
print(df[[
    "maquina", "temperatura", "severidade",
    "criticidade_ativo", "score_prioridade", "prioridade"
]])

In [ ]:
styled_df = df.head().style

# Definindo a cor de fundos, a cor do texto, e a cor do DataFrame
styled_df.set_properties(**{"background-color": "#254E58", "color": "#e9c46a", "border": "1.5px solid black"})

# Modificando a cor de fundo dos cabeçalhos da tabela
styled_df.set_table_styles([
    {"selector": "th", "props": [("color", 'white'), ("background-color", "#333333")]}
])

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 5

In [ ]:
# 1. CONFIGURAÇÃO
fig = plt.figure(figsize=(16, 7), facecolor="#f7f7f5")
ax = fig.add_axes([0, 0, 1, 1])
ax.set(xlim=(0, 16), ylim=(0, 7))
ax.axis("off")

azul_escuro, verde, laranja, vermelho, azul = (
    "#15284f", "#48b8ad", "#f2aa32", "#ff6b50", "#4d6bd8"
)


def card(x, y, w, h, cor="white", r=0.16):
    ax.add_patch(FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.02,rounding_size={r}",
        linewidth=0, facecolor=cor
    ))


# 2. ÁREA PRINCIPAL
card(0.15, 0.15, 15.7, 6.6, r=0.25)
ax.text(0.65, 6.35, "KPIs EM TEMPO REAL",
        fontsize=18, fontweight="bold", color=azul_escuro)

# 3. KPIs
kpis = [
    ("DISPONIBILIDADE MÉDIA", "4,2°C", verde, "↗"),
    ("DISPOSITIVOS ATIVOS", "48 / 50", vermelho, "↗"),
    ("ALERTAS HOJE", "7", laranja, "↓"),
    ("UPTIME", "98,4%", azul, "↗")
]

for i, (titulo, valor, cor, seta) in enumerate(kpis):
    x = 0.65 + i * 1.75
    card(x, 4.15, 1.55, 2.02)
    card(x, 5.92, 1.55, 0.25, cor, 0.12)

    ax.text(x + 1.15, 5.48, seta, fontsize=30, color=cor,
            ha="center", va="center", fontweight="bold")
    ax.text(x + 0.18, 4.78, titulo, fontsize=7.5, color="#6c7078")
    ax.text(x + 0.18, 4.42, valor, fontsize=19,
            fontweight="bold", color=azul_escuro)

# 4. TEMPERATURA
card(8.1, 4.15, 7.1, 2.02)
ax.text(8.35, 6.02, "TEMPERATURA POR CÂMARA (ÚLTIMAS 24H)",
        fontsize=11, fontweight="bold", color=azul_escuro)

ax_temp = fig.add_axes([0.53, 0.61, 0.39, 0.20])

x = np.arange(20)
y = np.array([
    2.6, 2.5, 2.4, 2.5, 2.7, 2.8, 2.7, 2.6, 2.4, 2.4,
    2.4, 2.6, 2.8, 3.1, 3.2, 3.3, 3.5, 3.6, 3.5, 3.5
])

ax_temp.plot(x, y, color=verde, linewidth=2.5)
ax_temp.fill_between(x, y, 1.9, color=verde, alpha=0.08)
ax_temp.axhline(2.5, color=vermelho, linestyle="--", linewidth=1.4)
ax_temp.set_ylim(1.8, 4.0)
ax_temp.set_xticks([])
ax_temp.set_yticks([])
ax_temp.grid(alpha=0.08)

for s in ax_temp.spines.values():
    s.set_visible(False)

# 5. UPTIME
card(0.65, 0.55, 7.0, 3.2)
ax.text(0.9, 3.45, "UPTIME POR GRUPO DE SENSORES",
        fontsize=11, fontweight="bold", color=azul_escuro)

ax_bar = fig.add_axes([0.07, 0.18, 0.39, 0.26])

grupos = ["CÂMARA 1", "CÂMARA 2", "CÂMARA B", "CÂMARA C", "GRUPO D", "GRUPO E"]
valores = [95, 78, 97, 82, 92, 90]
cores = [verde, laranja, vermelho, azul, laranja, azul]

ax_bar.bar(np.arange(len(grupos)), valores, color=cores, width=0.42)
ax_bar.set_ylim(0, 110)
ax_bar.set_xticks(np.arange(len(grupos)), grupos, fontsize=7)
ax_bar.set_yticks([])
ax_bar.grid(axis="y", alpha=0.08)

for s in ax_bar.spines.values():
    s.set_visible(False)

# 6. ALERTAS
card(8.0, 0.55, 7.2, 3.2)
ax.text(8.3, 3.45, "ALERTAS POR SEVERIDADE",
        fontsize=11, fontweight="bold", color=azul_escuro)

ax_pie = fig.add_axes([0.55, 0.18, 0.18, 0.22])
ax_pie.axis("equal")
ax_pie.axis("off")

sizes = [35, 25, 25, 15]
colors = [azul, vermelho, laranja, verde]
start = 90

for size, cor in zip(sizes, colors):
    theta = 360 * size / 100
    ax_pie.add_patch(Wedge(
        (0, 0), 1, start - theta, start,
        width=0.38, facecolor=cor, edgecolor="white"
    ))
    start -= theta

legendas = [
    ("CRÍTICO 35%", azul),
    ("ALTO 25%", vermelho),
    ("MÉDIO 25%", laranja),
    ("BAIXO 15%", verde)
]

for i, (txt, cor) in enumerate(legendas):
    y = 2.35 - i * 0.32
    ax.add_patch(plt.Rectangle((12.1, y), 0.11, 0.11, color=cor))
    ax.text(12.35, y, txt, fontsize=9, color="#42464d")

plt.show()

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 6

In [ ]:
# 1. CONFIGURAÇÃO
fig = plt.figure(figsize=(14, 7), facecolor="#f4f4f2")
ax = fig.add_axes([0, 0, 1, 1])
ax.set(xlim=(0, 14), ylim=(0, 7))
ax.axis("off")

VERMELHO, AZUL, LARANJA = "#f0443e", "#2f9ed3", "#f5a23a"
AZUL_ESCURO, VERDE, TEXTO = "#07264a", "#45b8ad", "#0c2545"


def card(x, y, w, h, cor="white", raio=0.18):
    ax.add_patch(FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.02,rounding_size={raio}",
        facecolor=cor, edgecolor="none"
    ))


# 2. ÁREA PRINCIPAL
card(0.15, 0.15, 13.7, 6.7, raio=0.22)

# 3. KPIs
kpis = [
    ("TEMPERATURA MÉDIA", "28.4°C", VERMELHO),
    ("UPTIME DOS DISPOSITIVOS", "98.7%", AZUL),
    ("ALERTAS HOJE", "12", LARANJA),
    ("CONSUMO ENERGÉTICO", "245 kWh", AZUL_ESCURO)
]

for i, (titulo, valor, cor) in enumerate(kpis):
    x = 0.7 + i * 3.2
    card(x, 5.35, 2.75, 1.15, cor, 0.12)
    ax.text(x + 0.18, 6.10, titulo, fontsize=8, color="white", fontweight="bold")
    ax.text(x + 0.18, 5.65, valor, fontsize=20, color="white", fontweight="bold")

# 4. MONITORAMENTO
card(0.65, 0.65, 6.1, 4.2, "#fafafa", 0.12)
ax.text(0.95, 4.52, "MONITORAMENTO EM TEMPO REAL",
        fontsize=11, color=TEXTO, fontweight="bold")

ax_mon = fig.add_axes([0.09, 0.18, 0.34, 0.38])
x = np.array([0, 25, 50, 75, 100])

ax_mon.plot(x, [12, 13, 18, 15, 48], color=VERMELHO, linewidth=2.5, label="Temperatura")
ax_mon.plot(x, [7, 11, 13, 12, 9], color=AZUL, linewidth=2.5, label="Umidade")

ax_mon.set_title("Leituras dos Sensores", fontsize=10)
ax_mon.set_xlabel("Tempo")
ax_mon.set_ylabel("Leituras")
ax_mon.grid(alpha=0.2)
ax_mon.legend()

for s in ax_mon.spines.values():
    s.set_color("#dddddd")

# 5. STATUS DOS DISPOSITIVOS
card(7.15, 0.65, 6.1, 4.2, "#fafafa", 0.12)
ax.text(7.45, 4.52, "STATUS DOS DISPOSITIVOS",
        fontsize=11, color=TEXTO, fontweight="bold")

ax_pie = fig.add_axes([0.57, 0.20, 0.22, 0.32])
ax_pie.pie(
    [87, 8, 5],
    colors=[VERDE, LARANJA, VERMELHO],
    startangle=90,
    counterclock=False,
    wedgeprops={"width": 0.35, "edgecolor": "white"}
)
ax_pie.axis("equal")
ax_pie.axis("off")

# 6. LEGENDA
legenda = [
    ("OPERACIONAIS 87%", VERDE),
    ("EM MANUTENÇÃO 8%", LARANJA),
    ("OFFLINE 5%", VERMELHO)
]

for i, (texto, cor) in enumerate(legenda):
    y = 1.70 - i * 0.35
    ax.add_patch(plt.Rectangle((10.0, y), 0.16, 0.16, color=cor))
    ax.text(10.25, y, texto, fontsize=10, color="#333333")

plt.show()